## Phase 2 — Grade Claims and Tag Severity (CPU + OpenAI)

### Cell 1 — Setup

In [ ]:
!pip -q install openai scikit-learn
import os, sys; sys.path.insert(0, ".")
os.environ["OPENAI_API_KEY"] = "<from Kaggle Secret / Colab userdata>"  # do NOT hard-code in shared repo
from sac.kqa_loader import load_kqa, gold_statements
from sac.cache import load_claims, append_claims, existing_claim_ids
from sac.grader import grade_claim, tag_severity
from sac.openai_judge import OpenAIJudge

### Cell 2 — Load inputs + build gold lookup

In [ ]:
items = {it.qid: it for it in load_kqa("questions.jsonl")}
claims = load_claims("claims_phase1.jsonl")
judge = OpenAIJudge(model="gpt-4o")
OUT = "claims_phase2.jsonl"
done = existing_claim_ids(OUT)

### Cell 3 — Idempotent grade + severity loop (one row appended per claim, resumable)

In [ ]:
for c in claims:
    if c.claim_id in done:
        continue
    statements = gold_statements(items[c.answer_id])
    c.label, c.grader_rationale = grade_claim(c.text, statements, judge)
    c.tier, c.severity_rationale = tag_severity(c.text, judge)
    append_claims(OUT, [c])                         # checkpoint per claim
    done.add(c.claim_id)
print("graded:", len(load_claims(OUT)))

### Cell 4 — Quick distribution sanity

In [ ]:
graded = load_claims(OUT)
from collections import Counter
print("labels:", Counter(c.label for c in graded))   # expect mix of 0/1/-1
print("tiers :", Counter(c.tier for c in graded))     # expect dangerous/benign